In [ ]:
import pandas as pd
from pathlib import Path

# K-Means Clustering Analysis

Run k-means clustering on each dataset and save cluster assignments for ARI analysis.

In [ ]:
from sklearn.cluster import KMeans
import numpy as np

# Directory with cleaned input files
input_dir = Path("../data/regional_data")

# Directory for cluster results
cluster_output_dir = Path("../results/05_kmeans/kmeans_clusters")
cluster_output_dir.mkdir(parents=True, exist_ok=True)

print(f"Input directory: {input_dir}")
print(f"Output directory: {cluster_output_dir}")

In [ ]:
# Get all CSV/TSV files from directory
input_files = sorted(input_dir.glob("*.csv")) + sorted(input_dir.glob("*.tsv"))

print(f"Found {len(input_files)} files:")
for f in input_files:
    print(f"  - {f.name}")

In [ ]:
# K-means parameters
n_clusters = 4  # Adjust this value as needed
random_state = 42  # For reproducibility

# ID column name
id_column = 'Id'

print(f"K-means settings:")
print(f"  Number of clusters: {n_clusters}")
print(f"  Random state: {random_state}")
print(f"  ID column: {id_column}")

In [ ]:
# Process each dataset
all_results = {}

for input_file in input_files:
    print(f"\n{'='*60}")
    print(f"Processing: {input_file.name}")
    print('='*60)
    
    # Read the data
    df = pd.read_csv(input_file, sep='\t' if input_file.suffix == '.tsv' else ',', encoding='utf-8')
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
    
    # Get IDs
    ids = df[id_column].copy()
    
    # Select numeric columns for clustering (excluding ID column)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if id_column in numeric_cols:
        numeric_cols.remove(id_column)
    
    print(f"  Using {len(numeric_cols)} numeric features for clustering")
    
    # Prepare features (already scaled, no need for StandardScaler)
    X = df[numeric_cols].copy()
    
    # Handle missing values (fill with column mean)
    if X.isnull().any().any():
        print(f"  Warning: Found missing values, filling with column means")
        X = X.fillna(X.mean())
    
    # Run K-means (data is already scaled)
    print(f"  Running K-means with k={n_clusters}...")
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    cluster_labels = kmeans.fit_predict(X)
    
    # Create results dataframe
    results_df = pd.DataFrame({
        'Id': ids,
        'cluster': cluster_labels
    })
    
    # Save results
    output_filename = input_file.stem + '_clusters.tsv'
    output_path = cluster_output_dir / output_filename
    results_df.to_csv(output_path, sep='\t', index=False, encoding='utf-8')
    
    print(f"  ✓ Saved cluster assignments to: {output_filename}")
    print(f"  Cluster distribution:")
    for cluster_id in sorted(results_df['cluster'].unique()):
        count = (results_df['cluster'] == cluster_id).sum()
        print(f"    Cluster {cluster_id}: {count} items")
    
    # Store in memory for further analysis
    all_results[input_file.stem] = results_df

print(f"\n{'='*60}")
print(f"✓ All datasets processed successfully!")
print(f"Results saved to: {cluster_output_dir}")
print('='*60)

# Multiple K-Means Runs

Run k-means multiple times with different random initializations and save all results for ARI analysis.

In [ ]:
# Parameters for multiple runs
n_runs = 100  # Number of k-means runs (variable)
n_clusters = 4  # Number of clusters
id_column = 'Id'

# Output directories
multi_run_dir = Path("../results/05_kmeans/kmeans_multiple_runs")
multi_run_dir.mkdir(parents=True, exist_ok=True)

centroids_dir = Path("../results/05_kmeans/kmeans_centroids")
centroids_dir.mkdir(parents=True, exist_ok=True)

print(f"Multiple runs settings:")
print(f"  Number of runs: {n_runs}")
print(f"  Number of clusters: {n_clusters}")
print(f"  Cluster results: {multi_run_dir}")
print(f"  Centroids: {centroids_dir}")

In [ ]:
# Process each dataset with multiple runs
for input_file in input_files:
    print(f"\n{'='*60}")
    print(f"Processing: {input_file.name}")
    print(f"Running {n_runs} k-means iterations...")
    print('='*60)
    
    # Read the data
    df = pd.read_csv(input_file, sep='\t' if input_file.suffix == '.tsv' else ',', encoding='utf-8')
    
    # Get IDs
    ids = df[id_column].copy()
    
    # Select numeric columns for clustering (excluding ID column)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if id_column in numeric_cols:
        numeric_cols.remove(id_column)
    
    # Prepare features (already scaled)
    X = df[numeric_cols].copy()
    
    # Handle missing values
    if X.isnull().any().any():
        X = X.fillna(X.mean())
    
    # Store all centroids for this dataset
    all_centroids = []
    
    # Run k-means n_runs times
    for run_idx in range(n_runs):
        # Run K-means with different random state for each run
        kmeans = KMeans(n_clusters=n_clusters, random_state=run_idx, n_init=10)
        cluster_labels = kmeans.fit_predict(X)
        
        # Save cluster assignments for this run
        results_df = pd.DataFrame({
            'Id': ids,
            'cluster': cluster_labels
        })
        
        # Save to file: dataset_name_run_XXX_clusters.tsv
        output_filename = f"{input_file.stem}_run_{run_idx:03d}_clusters.tsv"
        output_path = multi_run_dir / output_filename
        results_df.to_csv(output_path, sep='\t', index=False, encoding='utf-8')
        
        # Store centroids for this run
        centroids_with_metadata = pd.DataFrame(kmeans.cluster_centers_, columns=numeric_cols)
        centroids_with_metadata.insert(0, 'run', run_idx)
        centroids_with_metadata.insert(1, 'cluster', range(n_clusters))
        all_centroids.append(centroids_with_metadata)
        
        # Progress indicator
        if (run_idx + 1) % 10 == 0:
            print(f"  Completed {run_idx + 1}/{n_runs} runs")
    
    # Save all centroids to a single file
    all_centroids_df = pd.concat(all_centroids, ignore_index=True)
    centroids_filename = f"{input_file.stem}_all_centroids.tsv"
    centroids_path = centroids_dir / centroids_filename
    all_centroids_df.to_csv(centroids_path, sep='\t', index=False, encoding='utf-8')
    
    print(f"  ✓ Saved {n_runs} cluster assignments to: {multi_run_dir}")
    print(f"  ✓ Saved all centroids to: {centroids_filename}")
    print(f"     Centroids shape: {all_centroids_df.shape} ({n_runs} runs × {n_clusters} clusters)")

print(f"\n{'='*60}")
print(f"✓ All datasets processed with {n_runs} runs!")
print(f"Cluster assignments: {multi_run_dir}")
print(f"Centroids: {centroids_dir}")
print('='*60)

# Adjusted Rand Index (ARI) Analysis

Compute pairwise ARI values for all k-means runs to assess clustering stability.

In [ ]:
# Import ARI utility function
import sys
sys.path.insert(0, '../src')

from nbra.ari_test import (
    compute_pairwise_ari
)

# Output directory for ARI results
ari_output_dir = Path("../results/05_kmeans/kmeans_ari")
ari_output_dir.mkdir(parents=True, exist_ok=True)

print("ARI analysis utilities imported successfully")
print(f"ARI output directory: {ari_output_dir}")

In [ ]:
# Compute ARI for each dataset
ari_results = {}

for input_file in input_files:
    dataset_name = input_file.stem
    print(f"\n{'='*60}")
    print(f"Computing ARI for: {dataset_name}")
    print('='*60)
    
    # Load all run results for this dataset
    run_files = sorted(multi_run_dir.glob(f"{dataset_name}_run_*_clusters.tsv"))
    print(f"  Found {len(run_files)} run files")
    
    # Load all runs into a list of DataFrames
    runs_data = []
    for run_file in run_files:
        df = pd.read_csv(run_file, sep='\t', encoding='utf-8')
        runs_data.append(df)
    
    # Compute pairwise ARI matrix (using 'cluster' column name)
    print(f"  Computing pairwise ARI matrix...")
    ari_matrix = compute_pairwise_ari(runs_data, cluster_column='cluster')
    
    # Save ARI matrix
    ari_matrix_file = ari_output_dir / f"{dataset_name}_ari_matrix.tsv"
    pd.DataFrame(ari_matrix).to_csv(ari_matrix_file, sep='\t', index=False, encoding='utf-8')
    print(f"  ✓ Saved ARI matrix to: {ari_matrix_file.name}")
    
    # Compute summary statistics
    upper_tri_indices = np.triu_indices_from(ari_matrix, k=1)
    ari_values = ari_matrix[upper_tri_indices]
    
    stats = {
        'dataset': dataset_name,
        'n_runs': len(runs_data),
        'ari_mean': np.mean(ari_values),
        'ari_std': np.std(ari_values),
        'ari_median': np.median(ari_values),
        'ari_min': np.min(ari_values),
        'ari_max': np.max(ari_values)
    }
    
    ari_results[dataset_name] = {
        'matrix': ari_matrix,
        'stats': stats
    }
    
    # Display summary
    print(f"  ARI Statistics:")
    print(f"    Mean: {stats['ari_mean']:.4f}")
    print(f"    Std:  {stats['ari_std']:.4f}")
    print(f"    Min:  {stats['ari_min']:.4f}")
    print(f"    Max:  {stats['ari_max']:.4f}")

print(f"\n{'='*60}")
print(f"✓ ARI analysis completed for all datasets!")
print('='*60)

In [ ]:
# Create summary table for all datasets
summary_data = []
for dataset_name, results in ari_results.items():
    stats = results['stats']
    summary_data.append(stats)

summary_df = pd.DataFrame(summary_data)
summary_df = summary_df[['dataset', 'n_runs', 'ari_mean', 'ari_std', 'ari_median', 'ari_min', 'ari_max']]

# Save summary table
summary_file = ari_output_dir / "ari_summary_all_datasets.tsv"
summary_df.to_csv(summary_file, sep='\t', index=False, encoding='utf-8')

print("\nARI Summary for All Datasets:")
print("="*80)
print(summary_df.to_string(index=False))
print("="*80)
print(f"\n✓ Saved summary table to: {summary_file}")


# Elbow Method Analysis for Optimal K Selection

Determine the optimal number of clusters for each region using:
1. **Elbow Method**: Plot inertia (within-cluster sum of squares) vs K

Each input file represents a different region in the study.

In [ ]:
# Import required libraries for elbow analysis
import matplotlib.pyplot as plt
import matplotlib.cm as cm

# Create output directory for K-selection results
k_selection_dir = Path("../results/05_kmeans/k_selection")
k_selection_dir.mkdir(parents=True, exist_ok=True)

print("Libraries imported successfully")
print(f"K-selection output directory: {k_selection_dir}")

In [ ]:
# Parameters for K-selection
k_range = range(2, 11)  # Test K from 2 to 10
random_state = 42

# Storage for results
elbow_results = {}

print(f"Testing K values from {min(k_range)} to {max(k_range)}")
print(f"Number of regions to analyze: {len(input_files)}")

In [ ]:
# Run elbow method analysis for each region
for input_file in input_files:
    region_name = input_file.stem
    print(f"\n{'='*70}")
    print(f"Analyzing region: {region_name}")
    print('='*70)
    
    # Read the data
    df = pd.read_csv(input_file, sep='\t' if input_file.suffix == '.tsv' else ',', encoding='utf-8')
    
    # Select numeric columns for clustering (excluding ID column)
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if id_column in numeric_cols:
        numeric_cols.remove(id_column)
    
    # Prepare features
    X = df[numeric_cols].copy()
    
    # Handle missing values
    if X.isnull().any().any():
        X = X.fillna(X.mean())
    
    print(f"  Dataset shape: {X.shape}")
    print(f"  Features used: {len(numeric_cols)}")
    
    # Initialize storage for this region
    inertias = []
    
    # Test different K values
    for k in k_range:
        # Run K-means
        kmeans = KMeans(n_clusters=k, random_state=random_state, n_init=10)
        cluster_labels = kmeans.fit_predict(X)
        
        # Store inertia (for elbow method)
        inertias.append(kmeans.inertia_)
        
     
        
        print(f"  K={k}: Inertia={kmeans.inertia_:.2f}")
    
    # Store results
    elbow_results[region_name] = {
        'k_values': list(k_range),
        'inertias': inertias
    }
    
    

print(f"\n{'='*70}")
print("✓ Elbow analysis completed for all regions!")
print('='*70)

In [ ]:
# Save results to TSV files
for region_name in elbow_results.keys():
    # Save elbow results
    elbow_df = pd.DataFrame({
        'K': elbow_results[region_name]['k_values'],
        'Inertia': elbow_results[region_name]['inertias']
    })
    elbow_file = k_selection_dir / f"{region_name}_elbow.tsv"
    elbow_df.to_csv(elbow_file, sep='\t', index=False, encoding='utf-8')

print(f"✓ Saved elbow results for all regions to: {k_selection_dir}")

In [ ]:
# Visualize Elbow Method for all regions
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, (region_name, results) in enumerate(elbow_results.items()):
    ax = axes[idx]
    ax.plot(results['k_values'], results['inertias'], 'bo-', linewidth=2, markersize=8)
    ax.set_xlabel('Number of Clusters (K)', fontsize=11)
    ax.set_ylabel('Inertia (Within-cluster sum of squares)', fontsize=11)
    ax.set_title(f'Elbow Method: {region_name}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_xticks(results['k_values'])

plt.tight_layout()
elbow_plot_file = k_selection_dir / "elbow_method_all_regions.png"
plt.savefig(elbow_plot_file, dpi=300, bbox_inches='tight')
print(f"✓ Saved elbow plot to: {elbow_plot_file}")
plt.show()